In [1]:
# Core ML models
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

# Utilities
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd


In [2]:
# Use raw string (r"") to avoid escape character issues in Windows paths
df = pd.read_csv(r"data/dataset.csv")

# View the first few rows
print(df.head())


         y     x1     x2  x3     x4      x5     x6       x7  x8      x9
0  680.989  1.371 -2.694   0  7.560   6.645  1.786  119.128  10  14.205
1  521.047 -0.565  1.199   0  7.653  13.223  1.444  107.897   7  20.875
2  574.139  0.363 -0.716   0  7.280  16.861  0.312  110.826   2  24.140
3  515.591  0.633 -1.276   0  7.980   4.891  1.420  122.313  10  12.871
4  517.828  0.404 -0.806   0  7.644   8.478  0.982  111.969   8  16.122


In [3]:
# Split dataset into training (0-99) and test (100-119)
train_df = df.iloc[:100].dropna()
test_df = df.iloc[100:].drop(columns='y')

In [4]:
# Features and target
X_train = train_df.drop(columns='y')
y_train = train_df['y']
X_test = test_df.copy()

In [5]:
# Define all models in a dictionary
all_models = {
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Ridge Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5))
    ]),
    'Lasso Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LassoCV(cv=5,max_iter=10000, tol=1e-4))
    ]),
    'ElasticNet Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', ElasticNetCV(cv=5,max_iter=10000))
    ]),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0),
    'SVR': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVR())
    ])
}

In [6]:
# Fit and evaluate all models
final_results = {}

In [7]:
for name, model in all_models.items():
    model.fit(X_train, y_train)
    cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    train_preds = model.predict(X_train)
    rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    final_results[name] = {'CV R2': round(cv_r2, 4), 'RMSE': round(rmse, 2)}

final_results_df = pd.DataFrame(final_results).T

In [8]:
print(final_results_df)

                           CV R2      RMSE
Linear Regression         0.8186     78.45
Ridge Regression          0.7248    108.26
Lasso Regression          0.6095    167.08
ElasticNet Regression -1809.0023  56650.05
Random Forest             0.6737  15562.62
XGBoost                   0.8149    296.12
SVR                      -0.1420  61703.85


Best Model predictions and coefficients

In [9]:
# Fit Linear Regression with StandardScaler pipeline
linear_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('linear', LinearRegression())
])

# Train on rows 0–99 (first 100 rows)
linear_pipeline.fit(X_train, y_train)

# Predict Y for rows 101–120
y_pred_linear = linear_pipeline.predict(X_test)

# Extract model coefficients and intercept
linear_model = linear_pipeline.named_steps['linear']
coefficients = linear_model.coef_
intercept = linear_model.intercept_

# Prepare predicted values for display
linear_predictions_df = pd.DataFrame({
    "Row": list(range(101, 121)),
    "Predicted_Y": y_pred_linear
})

# Display predictions for rows 101–120
print("Predicted Y values for rows 101–120:\n")
print(linear_predictions_df.to_string(index=False))

# Display model coefficients
print("\nLinear Regression Coefficients:\n")
for i, coef in enumerate(coefficients):
    print(f"  x{i+1}: {coef:,.2f}")

# Display intercept
print(f"\nIntercept: {intercept:,.2f}")



Predicted Y values for rows 101–120:

 Row  Predicted_Y
 101   735.078476
 102   668.199995
 103   277.988085
 104  1083.184522
 105   609.559249
 106   686.321010
 107   427.414903
 108   527.398934
 109   456.886582
 110   419.194212
 111   576.888862
 112   693.307834
 113   387.794749
 114   356.208792
 115    58.099994
 116   352.649890
 117   289.793617
 118   752.711119
 119   513.847825
 120   390.761030

Linear Regression Coefficients:

  x1: 41,345.24
  x2: 21,969.07
  x3: -15,636.64
  x4: -19,547,252.00
  x5: -26,435,276.98
  x6: 7,551.27
  x7: 35,305.38
  x8: 7,076.68
  x9: 45,314,087.04

Intercept: 10,253.16


Model: Linear regression
Scaling: Applied standard scalar before training to normalize input features
Notes: Coefficient magnitude suggest some features have outsized influence especially x4, x5 and x9. This can be normal due to scaling. but may warrant further investigation or regularization